# 05 — GraphSAGE + LSTM Baseline (MCMIPF on-the-fly)

## Purpose
Train a baseline spatio-temporal model:
- Spatial encoder: GraphSAGE over a PATCH×PATCH graph
- Temporal encoder: LSTM over the history window
- Target: GHI at t_target (6h ahead)

## Import and settings

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import time
import random
from dataclasses import dataclass
from functools import lru_cache
from typing import Tuple, List, Dict, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

### Paths

In [2]:
PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"
GROUND_DIR   = PROJECT_ROOT / "data" / "ground_aligned"
RUNS_ROOT    = PROJECT_ROOT / "runs" / "graphsage_lstm"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)

### Device

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


### Experiment

In [4]:
SITE = "uniandes"  # "uniandes" | "elpaso" | ...
PATCH = 16         # keep small for GraphSAGE baseline
BATCH_SIZE = 8
NUM_WORKERS = 4

EPOCHS = 30
LR = 1e-3

PATIENCE = 8
MIN_DELTA = 0.0

GRAD_CLIP_NORM = 1.0
USE_AMP = (DEVICE == "cuda")

# NPZ hour-file cache size:
HOUR_CACHE_MAXSIZE = 16 #128

## Load meta and manifest

In [5]:
SITE_DIR = DATASET_ROOT / SITE
assert SITE_DIR.exists(), f"Missing dataset directory: {SITE_DIR}"

with open(SITE_DIR / "dataset_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Loaded dataset_meta.json:")
print(json.dumps(meta, indent=2))

MCMIPF_ROOT = Path(meta["mcmipf_root"])

PATCHES_ROOT = PROJECT_ROOT / "data" / "patches_v1" / SITE / f"P{PATCH}"
assert PATCHES_ROOT.exists(), f"Missing patch store: {PATCHES_ROOT}"
print("PATCHES_ROOT:", PATCHES_ROOT)

FREQ_MIN = int(meta["freq_min"])
H = int(meta["horizon_steps"])
L = int(meta["history_steps"])

GRID_SIZE = int(meta["grid_size"])
SITE_CENTER = (int(meta["site_center_pix"]["row"]), int(meta["site_center_pix"]["col"]))

HALF = PATCH // 2
assert PATCH % 2 == 0, "PATCH must be even"
assert 0 <= SITE_CENTER[0] < GRID_SIZE and 0 <= SITE_CENTER[1] < GRID_SIZE

train_man = pd.read_parquet(SITE_DIR / "manifest_train.parquet")
val_man   = pd.read_parquet(SITE_DIR / "manifest_val.parquet")
test_man  = pd.read_parquet(SITE_DIR / "manifest_test.parquet")

print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

Loaded dataset_meta.json:
{
  "site": "uniandes",
  "grid_size": 256,
  "site_center_pix": {
    "row": 160,
    "col": 49
  },
  "freq_min": 10,
  "horizon_steps": 36,
  "history_steps": 12,
  "mcmipf_root": "/srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF",
  "notes": "Manifest-only dataset. Satellite patches are extracted on-the-fly by the model."
}
PATCHES_ROOT: /srv/projects/Proyecto_e_ladino/data/patches_v1/uniandes/P16
train: (54508, 5)
val:   (12407, 5)
test:  (12075, 5)


In [6]:
DEBUG = False
DEBUG_TRAIN_N = 4000
DEBUG_VAL_N   = 1200
DEBUG_TEST_N  = 1200

In [7]:
def _hist_len(x):
    if isinstance(x, str):
        x = json.loads(x)
    return len(x)

L_manifest = int(train_man["history_ts"].map(_hist_len).mode().iloc[0])
L_meta = int(meta["history_steps"])
if L_manifest != L_meta:
    print(f"[WARN] meta L={L_meta} but manifest L={L_manifest}. Using manifest L.")
L = L_manifest

[WARN] meta L=12 but manifest L=24. Using manifest L.


In [8]:
# print("meta history_steps:", meta["history_steps"])
# print("meta freq_min:", meta["freq_min"])
# print("meta horizon_steps:", meta["horizon_steps"])

### Reproducibility

In [9]:
SEED = int(meta.get("seed", 42))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("SEED:", SEED)

def seed_worker(worker_id: int) -> None:
    worker_seed = (SEED + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

SEED: 42


In [10]:
if DEBUG:
    train_man = train_man.sample(n=min(DEBUG_TRAIN_N, len(train_man)), random_state=SEED).reset_index(drop=True)
    val_man   = val_man.sample(n=min(DEBUG_VAL_N, len(val_man)), random_state=SEED).reset_index(drop=True)
    test_man  = test_man.sample(n=min(DEBUG_TEST_N, len(test_man)), random_state=SEED).reset_index(drop=True)

print("DEBUG:", DEBUG)
print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

DEBUG: False
train: (54508, 5)
val:   (12407, 5)
test:  (12075, 5)


## Persistence baseline (same manifests)
Uses $\hat{y}_{(t+H)} = ghi(t)$

In [11]:
ground_path = GROUND_DIR / f"ground_10min_utc_{SITE}.parquet"
assert ground_path.exists(), f"Missing ground parquet for {SITE}: {ground_path}"

ground = pd.read_parquet(ground_path)
assert "ghi" in ground.columns, "Ground dataset missing 'ghi'"
assert str(ground.index.tz) == "UTC", "Ground index must be UTC"

def eval_persistence(manifest: pd.DataFrame, ground_df: pd.DataFrame) -> Dict[str, float]:
    t_label = pd.to_datetime(manifest["t_label"], utc=True)
    y_true = manifest["y"].astype(float).to_numpy()
    y_hat = ground_df.reindex(t_label)["ghi"].to_numpy()  # y(t)

    mask = np.isfinite(y_true) & np.isfinite(y_hat)
    y_true = y_true[mask]
    y_hat  = y_hat[mask]

    rmse = float(np.sqrt(np.mean((y_true - y_hat) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_hat)))
    return {"n": int(mask.sum()), "rmse": rmse, "mae": mae}

baseline_train = eval_persistence(train_man, ground)
baseline_val   = eval_persistence(val_man, ground)
baseline_test  = eval_persistence(test_man, ground)

print("=== Persistence baseline ===")
print("train:", baseline_train)
print("val:  ", baseline_val)
print("test: ", baseline_test)

=== Persistence baseline ===
train: {'n': 54470, 'rmse': 419.8303181060779, 'mae': 281.04479528180656}
val:   {'n': 12406, 'rmse': 378.32913327097435, 'mae': 247.05014589714656}
test:  {'n': 12075, 'rmse': 389.517844176586, 'mae': 250.79533714285714}


## Target normalization

In [12]:
y_train = train_man["y"].astype(float).to_numpy()
Y_MEAN = float(np.mean(y_train))
Y_STD  = float(np.std(y_train) + 1e-6)

def norm_y(y: float) -> float:
    return (y - Y_MEAN) / Y_STD

def denorm_y(arr: np.ndarray) -> np.ndarray:
    return arr * Y_STD + Y_MEAN

print("Target stats (train):")
print("  mean:", Y_MEAN)
print("  std :", Y_STD)
print("  percentiles:", np.percentile(y_train, [0, 50, 90, 95, 99]).tolist())

Target stats (train):
  mean: 180.78839786820285
  std : 283.1064661153594
  percentiles: [0.0, 2.202, 628.0594000000003, 858.5459999999997, 1088.74274]


## Graph edges
8-neighborhood for PATCH×PATCH grid

In [13]:
def build_edge_index_8n(patch: int) -> torch.Tensor:
    edges = []
    def node_id(rr: int, cc: int) -> int:
        return rr * patch + cc

    for rr in range(patch):
        for cc in range(patch):
            u = node_id(rr, cc)
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if dr == 0 and dc == 0:
                        continue
                    r2 = rr + dr
                    c2 = cc + dc
                    if 0 <= r2 < patch and 0 <= c2 < patch:
                        v = node_id(r2, c2)
                        edges.append((u, v))

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()  # (2, E)
    return edge_index

EDGE_INDEX = build_edge_index_8n(PATCH)
N_NODES = PATCH * PATCH
print("EDGE_INDEX:", EDGE_INDEX.shape, "| N_NODES:", N_NODES)

EDGE_INDEX: torch.Size([2, 1860]) | N_NODES: 256


## MCMIPF utils
hour path, slot, patch extraction, node features

In [14]:
# def hour_path_for_timestamp(t: pd.Timestamp) -> Path:
#     ymd = t.strftime("%Y%m%d")
#     hh  = t.strftime("%H")
#     year  = t.strftime("%Y")
#     month = t.strftime("%m")
#     fname = f"{ymd}_{hh}_MCMIPF.npz"
#     return MCMIPF_ROOT / year / month / fname

# def slot_for_timestamp(t: pd.Timestamp) -> int:
#     return int(t.strftime("%M")) // 10  # 0..5

# def extract_patch(frame_c_hw: np.ndarray, center_rc: Tuple[int, int], patch: int) -> np.ndarray:
#     r0, c0 = center_rc
#     half = patch // 2
#     r1, r2 = r0 - half, r0 + half
#     c1, c2 = c0 - half, c0 + half

#     if (r1 < 0) or (c1 < 0) or (r2 > GRID_SIZE) or (c2 > GRID_SIZE):
#         raise ValueError(f"Patch exceeds bounds: rows [{r1},{r2}) cols [{c1},{c2})")

#     return frame_c_hw[:, r1:r2, c1:c2]  # (C, P, P)

# def patch_to_node_features(patch_c_pp: np.ndarray) -> np.ndarray:
#     # (C, P, P) -> (P*P, C)
#     C, P1, P2 = patch_c_pp.shape
#     x = np.transpose(patch_c_pp, (1, 2, 0)).reshape(P1 * P2, C).astype(np.float32)
#     x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
#     return x

In [15]:
# # Hour-file cache
# @lru_cache(maxsize=HOUR_CACHE_MAXSIZE)
# def load_hour_npz(path_str: str) -> np.ndarray:
#     path = Path(path_str)
#     with np.load(path) as data:
#         arr = data["mcmipf"].astype(np.float32)  # (6, 16, 256, 256)
#     return arr

In [16]:
# from torch.utils.data import get_worker_info

# def load_hour_npz_nocache(path_str: str) -> np.ndarray:
#     path = Path(path_str)
#     with np.load(path) as data:
#         return data["mcmipf"].astype(np.float32)  # (6, 16, 256, 256)

# @lru_cache(maxsize=HOUR_CACHE_MAXSIZE)
# def load_hour_npz_maincache(path_str: str) -> np.ndarray:
#     # cache ONLY in main process
#     return load_hour_npz_nocache(path_str)

# def load_hour_npz(path_str: str) -> np.ndarray:
#     # If we're in a DataLoader worker, do NOT cache (avoid RAM blow-up across workers)
#     if get_worker_info() is None:
#         return load_hour_npz_maincache(path_str)
#     return load_hour_npz_nocache(path_str)

### Patch store utils

In [17]:
from torch.utils.data import get_worker_info

def patch_path_for_timestamp(t: pd.Timestamp) -> Path:
    # /YYYY/MM/YYYYMMDD_HH_patch.npz
    ymd = t.strftime("%Y%m%d")
    hh  = t.strftime("%H")
    year  = t.strftime("%Y")
    month = t.strftime("%m")
    fname = f"{ymd}_{hh}_patch.npz"
    return PATCHES_ROOT / year / month / fname

def slot_for_timestamp(t: pd.Timestamp) -> int:
    return int(t.strftime("%M")) // 10  # 0..5

def patch_to_node_features(patch_c_pp: np.ndarray) -> np.ndarray:
    # (C, P, P) -> (P*P, C)
    C, P1, P2 = patch_c_pp.shape
    x = np.transpose(patch_c_pp, (1, 2, 0)).reshape(P1 * P2, C).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x

def load_patch_hour_npz_nocache(path_str: str) -> np.ndarray:
    path = Path(path_str)
    with np.load(path) as data:
        # saved as float16
        return data["patch"]  # (6, 16, P, P), float16

@lru_cache(maxsize=HOUR_CACHE_MAXSIZE)
def load_patch_hour_npz_maincache(path_str: str) -> np.ndarray:
    return load_patch_hour_npz_nocache(path_str)

def load_patch_hour_npz(path_str: str) -> np.ndarray:
    if get_worker_info() is None:
        return load_patch_hour_npz_maincache(path_str)
    return load_patch_hour_npz_nocache(path_str)

## Helpers

### Dataset: sequence of graphs (x_seq) + normalized target y

In [18]:
class GraphSeqDataset(Dataset):
    """
    Returns:
      x_seq: torch.FloatTensor (L, N, C=16)
      y:     torch.FloatTensor scalar (normalized)
    """
    def __init__(self, manifest: pd.DataFrame, site_center: Tuple[int,int]):
        self.man = manifest.reset_index(drop=True)
        self.site_center = site_center

    def __len__(self) -> int:
        return len(self.man)

    def __getitem__(self, i: int):
        row = self.man.iloc[i]
        y = norm_y(float(row["y"]))

        history_ts = row["history_ts"]
        if isinstance(history_ts, str):
            history_ts = json.loads(history_ts)

        xs = []
        # for ts_str in history_ts:
        #     t = pd.to_datetime(ts_str, utc=True)
        #     p = hour_path_for_timestamp(t)
        #     slot = slot_for_timestamp(t)

        #     tensor = load_hour_npz(str(p))
        #     frame = tensor[slot]  # (16, 256, 256)

        #     patch = extract_patch(frame, self.site_center, PATCH)  # (16, P, P)
        #     x = patch_to_node_features(patch)                      # (N, 16)
        #     xs.append(x)
        for ts_str in history_ts:
            t = pd.to_datetime(ts_str, utc=True)
            p = patch_path_for_timestamp(t)
            slot = slot_for_timestamp(t)

            tensor = load_patch_hour_npz(str(p))   # (6, 16, P, P)
            patch = tensor[slot]                   # (16, P, P)

            x = patch_to_node_features(patch)      # (N, 16)
            xs.append(x)


        x_seq = np.stack(xs, axis=0)  # (L, N, 16)
        return torch.from_numpy(x_seq), torch.tensor(y, dtype=torch.float32)

train_ds = GraphSeqDataset(train_man, SITE_CENTER)
val_ds   = GraphSeqDataset(val_man, SITE_CENTER)
test_ds  = GraphSeqDataset(test_man, SITE_CENTER)

print("datasets:", len(train_ds), len(val_ds), len(test_ds))

datasets: 54508 12407 12075


### Dataloaders

In [19]:
def make_loader(ds: Dataset, shuffle: bool) -> DataLoader:
    kwargs = dict(
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=(NUM_WORKERS > 0),
    )
    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = 4
    return DataLoader(ds, **kwargs)

train_loader = make_loader(train_ds, shuffle=True)
# val_loader   = make_loader(val_ds, shuffle=False)
# test_loader  = make_loader(test_ds, shuffle=False)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=(DEVICE == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=(DEVICE == "cuda")
)

### Metrics

In [20]:
def metrics_from_arrays(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    return {"n": int(mask.sum()), "rmse": rmse, "mae": mae}

def skill_score(rmse_model: float, rmse_persist: float) -> float:
    return float(1.0 - (rmse_model / (rmse_persist + 1e-12)))

@torch.no_grad()
def eval_model(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    ys, yhats = [], []
    for x_seq, y in loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)  # (B,L,N,C)
        y = y.to(DEVICE, non_blocking=True)

        yhat = model(x_seq, EDGE_INDEX.to(DEVICE))
        ys.append(y.detach().cpu().numpy())
        yhats.append(yhat.detach().cpu().numpy())

    y = np.concatenate(ys)
    yhat = np.concatenate(yhats)

    # denormalize to physical units
    y_phys = denorm_y(y)
    yhat_phys = denorm_y(yhat)

    return metrics_from_arrays(y_phys, yhat_phys)

## Vectorized GraphSAGE components 
disjoint batched graph


In [21]:
def batch_edge_index(edge_index: torch.Tensor, batch_size: int, num_nodes: int, device=None) -> torch.Tensor:
    """
    Replicate edge_index for each graph in the batch, offsetting node ids by b*num_nodes.
    edge_index: (2, E)
    returns: (2, E*B)
    """
    if device is None:
        device = edge_index.device
    edge_index = edge_index.to(device)

    E = edge_index.size(1)
    offsets = (torch.arange(batch_size, device=device) * num_nodes).view(-1, 1)  # (B,1)
    batched = edge_index.unsqueeze(0) + offsets.unsqueeze(-1)                    # (B,2,E)
    batched = batched.permute(1, 0, 2).reshape(2, batch_size * E)                # (2,B*E)
    return batched.contiguous()

class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_nei  = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # x: (N, F), edge_index: (2, E) with edges u->v
        src, dst = edge_index[0], edge_index[1]
        N, F = x.shape

        nei_sum = torch.zeros((N, F), device=x.device, dtype=x.dtype)
        nei_sum.index_add_(0, dst, x[src])

        nei_cnt = torch.zeros((N, 1), device=x.device, dtype=x.dtype)
        nei_cnt.index_add_(0, dst, torch.ones((dst.numel(), 1), device=x.device, dtype=x.dtype))

        nei_mean = nei_sum / (nei_cnt + 1e-6)
        out = self.lin_self(x) + self.lin_nei(nei_mean)
        return torch.relu(out)

class GraphSAGE_LSTM(nn.Module):
    def __init__(self, in_dim: int, hidden_g: int, hidden_t: int, out_dim: int = 1):
        super().__init__()
        self.g1 = GraphSAGELayer(in_dim, hidden_g)
        self.g2 = GraphSAGELayer(hidden_g, hidden_g)

        self.lstm = nn.LSTM(input_size=hidden_g, hidden_size=hidden_t, batch_first=True)

        self.head = nn.Sequential(
            nn.Linear(hidden_t, hidden_t),
            nn.ReLU(),
            nn.Linear(hidden_t, out_dim),
        )

    def forward(self, x_seq: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # x_seq: (B, L, N, C)
        B, Ls, N, C = x_seq.shape
        device = x_seq.device

        be = batch_edge_index(edge_index, batch_size=B, num_nodes=N, device=device)  # (2, B*E)

        embeds = []
        for t in range(Ls):
            x = x_seq[:, t].reshape(B * N, C)  # (B*N, C)
            h = self.g1(x, be)
            h = self.g2(h, be)
            h = h.view(B, N, -1)
            g = h.mean(dim=1)  # (B, hidden_g)
            embeds.append(g)

        z = torch.stack(embeds, dim=1)  # (B, L, hidden_g)
        out, _ = self.lstm(z)           # (B, L, hidden_t)
        last = out[:, -1]               # (B, hidden_t)
        yhat = self.head(last).squeeze(-1)  # (B,)
        return yhat

### Sanity check

In [22]:
# 1) Batch shapes
x_seq0, y0 = next(iter(train_loader))
print("Sanity batch x_seq:", x_seq0.shape, "y:", y0.shape)
assert x_seq0.ndim == 4, x_seq0.shape
assert x_seq0.shape[1] == L, (x_seq0.shape, L)
assert x_seq0.shape[2] == N_NODES, (x_seq0.shape, N_NODES)
assert x_seq0.shape[3] == 16, x_seq0.shape
assert y0.ndim == 1, y0.shape

# 2) Hour loading speed sample
row = train_man.iloc[0]
history_ts = row["history_ts"]
if isinstance(history_ts, str):
    history_ts = json.loads(history_ts)

t0 = time.time()
for ts_str in history_ts[:10]:
    t = pd.to_datetime(ts_str, utc=True)
    # p = hour_path_for_timestamp(t)
    # tensor = load_hour_npz(str(p))
    # _ = tensor[slot_for_timestamp(t)]
    p = patch_path_for_timestamp(t)
    tensor = load_patch_hour_npz(str(p))
    _ = tensor[slot_for_timestamp(t)]
    
print("10 frames load time (s):", round(time.time() - t0, 4))

Sanity batch x_seq: torch.Size([8, 24, 256, 16]) y: torch.Size([8])
10 frames load time (s): 0.0043


## Model
**GraphSAGE per frame + LSTM over time**

In [23]:
model = GraphSAGE_LSTM(in_dim=16, hidden_g=64, hidden_t=64).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)

loss_fn = nn.MSELoss()  # MSE in normalized space
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

n_params = sum(p.numel() for p in model.parameters())
print("Model params:", round(n_params / 1e6, 3), "M")

Model params: 0.048 M


/tmp/ipykernel_597118/4120099060.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [24]:
# model = GraphSAGE_LSTM(in_dim=16, hidden_g=64, hidden_t=64).to(DEVICE)
# opt = torch.optim.Adam(model.parameters(), lr=1e-3)
# loss_fn = nn.MSELoss()
# print("Model params:", sum(p.numel() for p in model.parameters())/1e6, "M")

### Sanity check

In [25]:
row = train_man.iloc[0]
history_ts = row["history_ts"]
if isinstance(history_ts, str):
    history_ts = json.loads(history_ts)

t0 = time.time()
for ts_str in history_ts[:10]:
    t = pd.to_datetime(ts_str, utc=True)
    # p = hour_path_for_timestamp(t)
    # tensor = load_hour_npz(str(p))
    # _ = tensor[slot_for_timestamp(t)]
    p = patch_path_for_timestamp(t)
    tensor = load_patch_hour_npz(str(p))
    _ = tensor[slot_for_timestamp(t)]

print("10 frames load time (s):", time.time() - t0)

10 frames load time (s): 0.0020399093627929688


## Training

In [ ]:
# ht = train_man.iloc[0]["history_ts"]

# if isinstance(ht, str):
#     ht = json.loads(ht)          # JSON string -> list
# elif isinstance(ht, np.ndarray):
#     ht = ht.tolist()             # ndarray -> list
# # else: assume it's already a list-like

# t0 = pd.to_datetime(ht[0], utc=True)

# pp = patch_path_for_timestamp(t0)
# print("sample patch file:", pp, pp.exists())

# with np.load(pp) as d:
#     arr = d["patch"]
# print("arr:", arr.shape, arr.dtype)

sample patch file: /srv/projects/Proyecto_e_ladino/data/patches_v1/uniandes/P16/2023/09/20230901_05_patch.npz True
arr: (6, 16, 16, 16) float16


In [ ]:
run_name = f"{SITE}_H{H}_L{L}_P{PATCH}_seed{SEED}_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = RUN_DIR / "best_model.pt"
SUMMARY_PATH = RUN_DIR / "summary.json"

train_log: List[Dict[str, Any]] = []
best_val_rmse = float("inf")
best_epoch = 0
bad_epochs = 0

t_train0 = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_losses = []

    for x_seq, y in train_loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            yhat = model(x_seq, EDGE_INDEX.to(DEVICE))
            loss = loss_fn(yhat, y)  # normalized MSE

        scaler.scale(loss).backward()

        if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        scaler.step(opt)
        scaler.update()

        tr_losses.append(loss.item())

    # --- Validation in PHYSICAL units ---
    val_metrics = eval_model(model, val_loader)
    val_rmse = float(val_metrics["rmse"])
    val_mae  = float(val_metrics["mae"])

    scheduler.step(val_rmse)

    epoch_out = {
        "epoch": epoch,
        "train_mse_norm": float(np.mean(tr_losses)),   # normalized space
        "val_rmse_phys": val_rmse,                     # physical units
        "val_mae_phys": val_mae,
        "lr": float(opt.param_groups[0]["lr"]),
        "epoch_seconds": float(time.time() - t0),
    }
    train_log.append(epoch_out)

    improved = (best_val_rmse - val_rmse) > MIN_DELTA

    if improved:
        best_val_rmse = val_rmse
        best_epoch = epoch
        bad_epochs = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "best_val_rmse_phys": best_val_rmse,
                "meta": {
                    "site": SITE,
                    "patch": PATCH,
                    "L": L,
                    "H": H,
                    "seed": SEED,
                    "y_mean_train": Y_MEAN,
                    "y_std_train": Y_STD,
                },
            },
            BEST_PATH,
        )
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_mse_norm={epoch_out['train_mse_norm']:.5f} | "
        f"val_rmse_phys={val_rmse:.2f} | val_mae_phys={val_mae:.2f} | "
        f"lr={epoch_out['lr']:.2e} | "
        f"time={epoch_out['epoch_seconds']:.1f}s | "
        f"best={best_val_rmse:.2f} (ep {best_epoch}) | "
        f"bad={bad_epochs}/{PATIENCE}"
    )

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch {best_epoch} val_rmse={best_val_rmse:.2f}")
        break

total_train_seconds = float(time.time() - t_train0)
print("Training finished. Total seconds:", round(total_train_seconds, 1))
print("Best checkpoint:", BEST_PATH)

### Load best checkpoint and evaluate (val/test) + skill score

In [ ]:
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

print("Loaded best model from epoch:", ckpt["epoch"], "| best_val_rmse_phys:", ckpt["best_val_rmse_phys"])

final_val = eval_model(model, val_loader)
final_test = eval_model(model, test_loader)

final_val["skill_vs_persistence"] = skill_score(final_val["rmse"], baseline_val["rmse"])
final_test["skill_vs_persistence"] = skill_score(final_test["rmse"], baseline_test["rmse"])

print("=== Final evaluation (best ckpt) ===")
print("Persistence val :", baseline_val)
print("Model val       :", final_val)
print("Persistence test:", baseline_test)
print("Model test      :", final_test)

## Summary 

In [ ]:
summary = {
    "run_name": run_name,
    "site": SITE,
    "device": DEVICE,
    "seed": SEED,
    "debug": {
        "enabled": bool(DEBUG),
        "train_n": int(len(train_man)),
        "val_n": int(len(val_man)),
        "test_n": int(len(test_man)),
    },
    "data_paths": {
        "site_dir": str(SITE_DIR),
        "mcmipf_root": str(MCMIPF_ROOT),
        "patches_root": str(PATCHES_ROOT),
        "ground_path": str(ground_path),
        "run_dir": str(RUN_DIR),
    },
    "temporal": {
        "freq_min": int(FREQ_MIN),
        "history_steps_L": int(L),
        "horizon_steps_H": int(H),
        "history_hours": float(L * FREQ_MIN / 60.0),
        "horizon_hours": float(H * FREQ_MIN / 60.0),
    },
    "spatial": {
        "grid_size": int(GRID_SIZE),
        "patch": int(PATCH),
        "site_center_rc": {"row": int(SITE_CENTER[0]), "col": int(SITE_CENTER[1])},
        "nodes": int(N_NODES),
        "edge_index_shape": [int(EDGE_INDEX.shape[0]), int(EDGE_INDEX.shape[1])],
    },
    "target_norm": {
        "y_mean_train": float(Y_MEAN),
        "y_std_train": float(Y_STD),
        "y_percentiles_train": np.percentile(y_train, [0, 50, 90, 95, 99]).tolist(),
    },
    "baselines": {
        "persistence_train": baseline_train,
        "persistence_val": baseline_val,
        "persistence_test": baseline_test,
    },
    "model": {
        "arch": "GraphSAGE(2) + mean pool + LSTM + MLP head (vectorized disjoint batch graph)",
        "in_dim": 16,
        "hidden_g": 64,
        "hidden_t": 64,
        "optimizer": "Adam",
        "lr_init": float(LR),
        "scheduler": "ReduceLROnPlateau(factor=0.5, patience=3)",
        "batch_size": int(BATCH_SIZE),
        "num_workers": int(NUM_WORKERS),
        "use_amp": bool(USE_AMP),
        "grad_clip_norm": float(GRAD_CLIP_NORM),
        "epochs_max": int(EPOCHS),
        "patience": int(PATIENCE),
        "min_delta": float(MIN_DELTA),
        "train_seconds_total": float(total_train_seconds),
        "best_ckpt_path": str(BEST_PATH),
        "best_epoch": int(best_epoch),
        "best_val_rmse_phys": float(best_val_rmse),
        "final_val": final_val,
        "final_test": final_test,
        "hour_cache_maxsize": int(HOUR_CACHE_MAXSIZE),
    },
    "training_log": train_log,
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved summary:", SUMMARY_PATH)